# Day 13(M3 Day04) 실습 1 — Naive RAG의 한계와 문서 전처리

**목표**: PDF를 그대로 읽어 만든 기본 RAG(Naive RAG)가 어디서 약한지 확인하고, PDF를 Markdown으로 바꿔 문서 구조(제목·표)를 살리면 무엇이 달라지는지 비교한다. Day13의 나머지 실습(질의 개선, 재순위·압축, 평가)은 여기서 만든 저장소를 쓴다.

| Part | 내용 |
| --- | --- |
| Part 1 | Naive RAG — PDF를 그대로 읽어 검색하고, 한계 세 가지를 확인한다 |
| Part 2 | PDF를 Markdown으로 바꿔 구조를 살리고, 같은 질문으로 비교한다 |   

| Day13 실습 | 주제 |
| --- | --- |
| **실습 1 (이 노트북)** | Naive RAG의 한계와 문서 전처리 |
| 실습 2 | 질의 개선 — 재작성, HyDE, Multi-Query, RAG Fusion |
| 실습 3 | 하이브리드 검색과 재순위 — BM25, 리랭킹, Flashrank, 문맥 압축 |
| 실습 4 | RAG 생성 품질 평가 |

**데이터**: 한국지능정보사회진흥원(NIA) AI@Data Report 2025-제3호 「토픽 분석을 통한 AI 주요 트렌드 및 2026 전망」(2025.12, 22쪽). 과학기술정보통신부·NIA '초거대 AI 데이터 발굴 및 기획' 사업의 산출물이다. 보고서 규정에 따라 무단전재를 금하고, 가공·인용할 때는 출처를 「한국지능정보사회진흥원(NIA)」으로 밝힌다. 저장소에는 넣지 않고 Day13-1이 NIA 공식 주소에서 `data/ai_trend_2026.pdf`로 내려받는다.   

> **참고:** `.env`의 `OPENAI_API_KEY`·`DATABASE_URL`(Supabase Session pooler)을 확인한다. `uv add pymupdf pymupdf4llm`이 필요하다(이미 설치돼 있다면 생략).

## 0. 환경 준비

In [1]:
import os, re
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

load_dotenv()
CONN = os.getenv("DATABASE_URL").replace("postgresql://", "postgresql+psycopg://")

# PGVector마다 연결 풀을 새로 만들지 않도록 엔진 하나를 같이 쓴다 (Session pooler는 연결 15개 한도)
engine = create_engine(CONN, pool_size=2, max_overflow=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)   # temperature=0: 결과 재현성

import urllib.request
import pymupdf, pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF = Path("data/ai_trend_2026.pdf")
if not PDF.exists():   # NIA 공식 주소에서 내려받는다 (저장소에는 없다)
    PDF.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://www.nia.or.kr/common/board/Download.do?bcIdx=28899&cbIdx=39485&fileNo=1", PDF)
print("PDF:", PDF, f"{PDF.stat().st_size / 1_000_000:.1f}MB")

PDF: data\ai_trend_2026.pdf 0.8MB


## Part 1. Naive RAG — PDF를 그대로 읽어 검색하기

Naive RAG는 **질문 → 검색 → 생성**을 한 번씩만 하는 가장 단순한 RAG다. Day12에서 만든 구조가 이것이다.

| 문제 | 원인 | 영향 |
| --- | --- | --- |
| 검색 품질이 흔들린다 | 질문 하나에만 의존 | 같은 뜻이라도 표현에 따라 결과가 달라진다 |
| 관련 없는 조각이 섞인다 | 관련이 없어도 k개를 무조건 가져온다 | 답변 품질이 떨어진다 |
| 표·목차가 깨진다 | PDF 텍스트를 줄 단위로 꺼낸다 | 표에 있는 사실을 찾지 못한다 |
| 지어낸 답이 나온다 | 검색이 실패해도 모델이 자기 지식으로 답한다 | 틀린 답 |

Day13 실습 2~4에서 이 문제를 하나씩 줄인다.

### 1-1. PDF를 그대로 읽어 저장

Day12 Part 4처럼 쪽마다 텍스트를 꺼내 자른다. 이번에는 정리 없이 그대로 쓴다. 조각마다 `chunk_id`를 붙여 두면 어느 조각이 검색됐는지 추적하기 쉽다.

In [2]:
# 문서 로딩
with pymupdf.open(PDF) as pdf:
    pdf_pages = [Document(page_content=p.get_text(), metadata={"source": PDF.name, "page": p.number + 1})
                 for p in pdf]
    
# TODO: 500자·겹침 80으로 pdf_pages를 메타데이터가 유지되도록 자르세요
# 청킹
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
pdf_chunks = splitter.split_documents(pdf_pages)

# metadata 추가하기
for i, c in enumerate(pdf_chunks, 1):
    c.metadata.update(chunk_id=i, doc_type="pdf")
    
# 벡터스토어 저장  collection_name = ai_trend_pdf
pdf_store = PGVector(embeddings=embeddings, collection_name="ai_trend_pdf",
                     connection=engine, use_jsonb=True, pre_delete_collection=True)
pdf_store.add_documents(pdf_chunks)

Collection not found


['f5fa34df-bcc3-4501-b642-bd2d94e49b38',
 'e17737fc-11de-4cea-b7d2-dd5ced28d60f',
 'ee18855d-d905-43bd-885b-7846c4731db7',
 'a73432ba-f161-4903-811c-f6b18897fc39',
 '0fdd9979-5281-43c1-8457-8283d9f945bb',
 '27ae307a-cf30-4187-9d01-9294b866f0a9',
 '42502ce1-d9db-4071-ba55-732fcc022ddb',
 'b6e3e16b-0bd4-4d76-87f3-18d6c54aa684',
 'a9f53dc6-5494-4cc4-89f5-476e7398d93c',
 '3eadc9b6-1937-45a4-96b1-a13d17fbabf7',
 '4f9f9653-234a-4d75-a92b-419e3cb351d2',
 '5079bfaa-ec1f-4bb5-a9dd-50e3c604b1f3',
 '04450dbf-1e67-4437-b4ce-6cf234f2669b',
 'c6d9fdac-a9b0-40aa-a205-b0f0c7c21553',
 'c50f0bf2-5ef2-4db9-90b1-4102019e51e6',
 '7fb9cc3b-dd68-4c8e-a76d-e5a538a0768f',
 '2209293b-636e-4ddf-bfd7-78b3ea64fb3d',
 '8a020e86-333a-4249-81e5-dd6caa2ffc8e',
 'ea3381c2-8e4e-4c28-9f8b-d9f965ced884',
 '43afd7fa-b6f5-486d-8cf6-08e51633bd9f',
 'd13adf50-96ae-419d-8bd8-05e5219856e9',
 '692063c6-75b4-42aa-bfa7-1d2be62aac34',
 'ef333fd4-94df-44e1-9b1e-c41aff5beb7d',
 'a3355b75-0abb-4f66-910e-c915d51a6f3c',
 'be1bbb8f-6e1f-

### 1-2. Naive RAG 체인

In [3]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

In [4]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서만 참고해 답한다. 문서에 없는 내용은 '문서에서 확인할 수 없습니다'라고 답한다.\n\n[문서]\n{context}"),
    ("human", "{question}"),
])

#래그 체인 구성 함수
def make_rag_chain(store, k=3):
    retriever = store.as_retriever(search_kwargs={"k": k})
    return {"context": retriever | format_docs, "question": RunnablePassthrough()} | answer_prompt | llm | StrOutputParser()

naive_chain = make_rag_chain(pdf_store)

print(naive_chain.invoke("2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"))

2026년 AI 기술 전망에서 핵심 변화는 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 것으로 예상되며, 합성데이터, 추론형 AI, 멀티모달 기술이 주요 경쟁 축으로 자리 잡을 것입니다. 이러한 변화는 학습 효율 향상, 복합 정보 처리, 설명 가능성 강화 등 모델 내부 구조의 질적 개선 흐름을 이끌 것으로 보입니다. 또한, 고품질 데이터 생성, 멀티모달, 고급 추론 기술이 결합되어 AI의 상황 이해와 문제 해결 능력이 향상되고, 서비스 및 산업 전반의 활용도도 확대될 전망입니다.


### 1-3. 한계 ① — 표현에 따라 검색 결과가 달라진다

같은 것을 묻는 질문 세 개로 검색한다. `similarity_search_with_score`의 점수는 **코사인 거리**라서 **낮을수록 가깝다**.

In [5]:
def inspect(store, query, k=3):
      # TODO: store에서 (문서, 거리) 쌍을 k개 받아 돌며 출력하세요 (similarity_search_with_score)
      print('질문 : ', query)
      
      for doc, dist in store.similarity_search_with_score(query, k=k):
            m = doc.metadata
            print(f"거리 : {dist:.3f} | {m['page']}쪽 {m['chunk_id']} 조각 | 원문 : {doc.page_content[:50]}")
      print()

In [6]:
for q in ["2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
          "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
          "합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?"]:
      inspect(pdf_store, q)

질문 :  2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
거리 : 0.359 | 18쪽 38 조각 | 원문 : 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
18
 
           
※ I
거리 : 0.367 | 5쪽 7 조각 | 원문 : 대응도 확대되는 흐름이 확인됨
■ 2026년 AI 생태계 방향성 전망
산업·기술·정책이 서
거리 : 0.389 | 19쪽 40 조각 | 원문 : 19
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
4.4
2026년 AI 정책 전

질문 :  내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
거리 : 0.426 | 18쪽 38 조각 | 원문 : 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
18
 
           
※ I
거리 : 0.448 | 19쪽 40 조각 | 원문 : 19
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
4.4
2026년 AI 정책 전
거리 : 0.448 | 1쪽 1 조각 | 원문 : AI@Data Report
                                   

질문 :  합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?
거리 : 0.413 | 21쪽 45 조각 | 원문 : 21
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
구분
설명
① 단계적 '과정' 
거리 : 0.427 | 20쪽 43 조각 | 원문 : 산재된 데이터를 모아 데이터의 One-Source Multi-Use 를 지원 
Ÿ AI 학
거리 : 0.437 | 18쪽 38 조각 | 원문 : 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
18
 
           
※ I



세 질문 모두 보고서 18쪽의 "2026년 AI 기술 전망"을 묻는다. 출력에서 질문마다 검색된 쪽과 거리가 어떻게 다른지 본다. 특히 "내년"처럼 보고서에 없는 표현을 쓰면 검색이 흔들린다. 실습 2의 **질의 재작성**이 이 문제를 다룬다.

### 1-4. 한계 ② — 표가 깨진다

보고서에는 표가 여러 개 있다. 칸이 2개인 단순한 표(10쪽 [표 2])와 칸이 4개인 복잡한 표(16쪽 [표 4])를 PDF 텍스트로 꺼내 보고, 표를 묻는 질문에 답하게 한다.

In [7]:
pdf_pages[9].page_content

'토픽 분석을 통한 AI 주요 트렌드 및 2026 전망\n10\nŸ 분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함\n본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심어 중 \n산업·기술·정책의 주요 이슈와 변화 방향을 가장 정확히 대표하는 단어들을 선별하여 구성\nŸ 핵심 키워드는 산업·기술·정책 각 분야에서 반복적으로 등장한 상위 빈도 핵심어 중 대표성이 \n가장 높은 단어를 기반으로 선정함\nŸ 이는 각 키워드가 분야별 담론 구조와 주요 이슈·변화 방향(시장 확산, 기술 고도화, 규제 강화 \n등)을 가장 명확하게 설명하는 지표 역할을 하기 때문임\n< 표 2 > 분야별 데이터 수집 키워드\n분야별로 설정한 7개 키워드를 기반으로 전체 수집 건수는 282건(분야별 94건)이며, 각 \n기사·리포트별로 주요 내용 요약본을 정리한 후, 산업·기술·정책별 세부 키워드 분석을 실시함\nŸ 해외 매체는 의미 왜곡을 최소화하고 분석 기준을 일관되게 유지하기 위해 한국어로 번역 후 \n요약하였으며, 이를 통해 분석 절차의 체계성과 비교 가능성을 확보함\n분야\n사용 키워드\nAI산업\nAI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환\nAI기술\nAI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발\nAI정책\nAI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책\n[ 그림 1 ] 분야별 데이터셋 구성 예시\n'

In [8]:
print(naive_chain.invoke("산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"))

산업·기술·정책 분야별 데이터 수집 키워드는 다음과 같습니다.

- AI산업: AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환
- AI기술: AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발
- AI정책: AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책


칸이 2개뿐인 [표 2]는 표 제목 뒤에 본문 문단이 끼어들었지만, 한 행의 내용(`AI산업` 다음 줄에 키워드 7개)이 붙어 있어서 답을 맞힌다. 이번에는 칸이 4개(분야, 토픽명, 주요 키워드, 핵심 이슈)인 [표 4]를 본다.

In [13]:
pdf_pages[15].page_content

'토픽 분석을 통한 AI 주요 트렌드 및 2026 전망\n16\n3.2.4\n[AI 산업·기술·정책 토픽] - 종합 결과 및 시사점\n< 표 4 > AI 산업·기술·정책 토픽 모델링 종합 결과 및 시사점\nAI 산업·기술·정책은 서로 다른 변화 축을 가지지만, 2025년을 전후한 산업의 확산, \n기술의 고도화, 정책의 제도화가 동시에 진행되는 복합적 전환기에 진입한 것으로 판단됨\n산업·기술·정책 변화의 핵심 축은 상이하나, AI 생태계는 ‘연결된 흐름’으로 작동함\nŸ 산업 확산은 기술 고도화를 요구하고, 기술의 고도화는 다시 정책적 대응을 촉발하는 \n연쇄적·단계적 상호작용 구조가 형성되어 있음\n세 분야는 ‘압력–수요–조정’의 순환 구조를 형성하며 상호 보완적으로 진화하고 있음\nŸ 산업 확산은 기술 고도화에 대한 수요를 높이고, 기술 고도화는 새로운 규제·기준 마련 필요성을 \n확대시키며, 이에 대한 정책 정비는 다시 산업·기술 영역에 조정 압력으로 작용하는 흐름이 나타남\nŸ 이는 AI 생태계가 일방향적 변화가 아니라, 산업–기술–정책이 서로 영향을 주고받는 \n순환적·상호작용적 시스템으로 진화하고 있음을 시사함\n이러한 변화는 AI를 단순 기술이 아닌 산업 인프라이자 경쟁·제도 관리의 핵심 요소로 부상시키고 \n있으며, 이에 따라 향후 AI 생태계의 지속성과 안정성을 위해 다음 과제가 요구됨\nŸ 산업확산–기술고도화–정책규제 간 속도 불일치를 완화하는 구조적 접근 필요\nŸ 정책적 대응 속도와 규제 체계의 예측 가능성 확보\nŸ 도입 확대·기술혁신·위험관리 간 균형 있는 추진 체계 확립\n분야\n토픽명\n주요 키워드\n핵심 이슈\nAI 산업\n시장 규모 \n확대와 비즈니스 \n구조 전환\n도입, 확대, 성장세, \n확산. 규모, 성장, \n글로벌, 분기, 비용, \n자금, 인프라, 센터, \n효율, 에이전트\nŸ AI 도입·확산 가속에 따른 산업 구조 변화\nŸ 산업 규모 확대 및 글로벌 경쟁 심화\nŸ 인프라·에이전트 기반 운영 모델 변화\nAI

In [14]:
print(naive_chain.invoke("AI 기술 분야 토픽모델링 결과에서 도출된 토픽들과 그 시사점은 무엇인가요?"))

AI 기술 분야 토픽모델링 결과에서 도출된 토픽은 다음과 같습니다:

1. **기능 고도화와 적용 범위 확장**: 
   - **지능 구조 고도화**: 기술 발전이 단순 성능 향상을 넘어 모델이 무엇을 이해하고 어떻게 추론하는지와 같은 지능 구조의 고도화로 이동하고 있음을 시사합니다. 
   - **성능 경쟁 심화**: 모델 개발 주기의 단축, 성능지표 향상, 효율·추론 능력 강화가 기술 경쟁력의 중심 요소가 되고 있음을 보여줍니다.

이러한 토픽들은 AI 기술이 복합 입력 처리, 고급 추론, 자연스러운 응답 능력 등 알고리즘 수준의 질적 확장을 통해 발전하고 있음을 나타냅니다.


[표 4]는 칸이 한 줄에 하나씩 흩어지고, 한 칸 안의 글도 여러 줄로 쪼개진다(`시장 규모` / `확대와 비즈니스` / `구조 전환`). 어느 글이 어느 열인지 표시가 사라진다.

출력에서 두 가지를 확인한다.

| 확인 | 이번 실행 결과 |
| --- | --- |
| 표가 있는 16쪽을 검색했나 | 검색된 쪽은 12·14·22쪽이다. 16쪽 표는 검색되지 않았다(2-4에서 확인) |
| 답이 표의 열을 구분하나 | 기술 분야 토픽은 1개인데, 핵심 이슈 열의 내용("성능 고도화와 개발·업데이트 주기의 가속")을 두 번째 토픽으로 답했다 |

> **참고:** 같은 설정이어도 실행에 따라 "문서에서 확인할 수 없습니다"로 답하기도 한다. 어느 쪽이든 표의 구조를 잃어서 생기는 문제다.

### 1-5. 한계 ③ — 관련이 없어도 k개를 가져온다

보고서에 없는 내용을 물어도 리트리버(Retriever)는 가장 가까운 조각 k개를 **항상** 돌려준다.

In [9]:
q = "한국의 올해 반도체 수출액은 얼마인가요?"
inspect(pdf_store, q)
naive_chain.invoke(q)

질문 :  한국의 올해 반도체 수출액은 얼마인가요?
거리 : 0.731 | 9쪽 17 조각 | 원문 : 9
토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
분석 절차
CHAPTER2
2.1
거리 : 0.743 | 22쪽 49 조각 | 원문 : 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
22
토픽 분석을 통한
AI 주요 트
거리 : 0.745 | 10쪽 19 조각 | 원문 : 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망
10
Ÿ 분야 간 키워드 수를 동일하



'문서에서 확인할 수 없습니다.'

### 관찰

| 한계 | 확인한 것 | 줄이는 방법 (Day13) |
| --- | --- | --- |
| ① 표현 편차 | 같은 뜻의 질문이 다른 조각을 가져온다 | 실습 2: 질의 재작성, Multi-Query |
| ② 표 깨짐 | 단순한 [표 2]는 맞히지만, 칸이 4개인 [표 4]는 표가 있는 쪽을 검색하지 못하고 열도 섞어 답한다 | Part 2: Markdown 변환 |
| ③ 무조건 k개 | 관련 없는 질문에도 조각 3개가 온다. 거리가 크다는 것만 다르다 | 실습 3: 재순위·문맥 압축 |

> **참고:** ③에서 답이 "문서에서 확인할 수 없습니다"로 나왔다면 프롬프트 지시 덕분이다. 검색 자체는 여전히 관련 없는 조각을 넘겼고, 그만큼 토큰을 썼다.

## Part 2. PDF를 Markdown으로 — 구조 살리기

`pymupdf4llm`은 PDF를 Markdown으로 바꾼다. 제목은 `#`로, 표는 `| 칸 | 칸 |` 형태로 남긴다. 구조가 남으면 표 한 행이 한 줄에 모이고, 조각을 잘라도 행 단위 정보가 유지된다.

### 2-1. Markdown으로 변환

In [10]:
# TODO: PDF를 쪽마다 Markdown으로 바꾸세요 (pymupdf4llm.to_markdown, page_chunks=True, table_strategy="lines")
md_pages = pymupdf4llm.to_markdown(str(PDF), page_chunks=True, table_strategy="lines")

In [28]:
md_pages[10]['text']

'11 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 \n\n###### **2.2 연구 방법** \n\n\n\n- 본 분석은 R Studio 4.5.2 환경과 텍스트 분석 패키지를 활용해 MeCab 형태소 분석기 및 명사 토큰화 알고리즘으로 원문을 구조화함 \n\n- Ÿ 전처리 과정에서는 숫자, 한 글자 단어, 조사, 불용어, 그리고 AI 문헌에서 빈번히 등장하는 단순 용어와 고유명사(국가명·기관명 등)를 체계적으로 제거함 \n\n**< 표 3 > 전처리 용어 정제 목록** \n\n|**분류**|**주요 예시들**|**특징**|\n|---|---|---|\n|숫자|0~9|분석에 불필요한 단어|\n|1음절<br>단어|수, 올, 등, 것 등|정보성 낮은 단어|\n|불용어|시점, 구분어, 접미/접두사|분석에 불필요한<br>일반어, 구분어|\n|국가<br>이름|한국, 대한민국, 미국, 일본, 중국, 유럽, 세계 등|지명, 국가명|\n|조사|의, 는, 에, 에서, 으로, 도, 와, 과 등|문법적 기능,<br>의미성 적음|\n|기타<br>불용어|관련, 기반, 중심, 수준, 부분, 측면, 방식, 올해, 내년, 작년,<br>최근, 리포트, 보고서, 발표, 기사, 보도, 분석, 딥시크, 진행,<br>달러, 전체, 기술전문지, 기술전문매체, 경제지, 통신사, 매체,<br>예정, 분야, 모델, 공개, 투자, 산업, 기술, 정책, 논의, 시장|빈번 반복, 맥락 무관,<br>문서·시점 단어|\n\n\n\n\n\n전처리 후, 정제된 텍스트 데이터를 활용해 LDA(Latent Dirichlet Allocation) 토픽모델링 을 수행함 \n\n- Ÿ LDA는 문서 집합에서 숨겨진 주제와 주요 키워드를 확률적으로 추출하는 대표적인 토픽 모델링 기법으로 대규모 텍스트에 내재된 담론 구조를 식별하는 데 효과적임<sup>3)</sup> \n\n- Ÿ 산업·기술·정책 각 분야 별 k값(토픽 수)를 2로 설정하였으나, 분석 결과의 타당성과 설명력을 종합적으로 고려해 최종 보고서에는 가장 

| 부분 | 의미 |
| --- | --- |
| `page_chunks=True` | 쪽마다 따로 변환해 목록으로 준다. 쪽 번호(`metadata["page_number"]`)가 남는다 |
| `table_strategy="lines"` | 선으로 그은 표를 찾아 Markdown 표로 바꾼다 |

1-4에서 흩어졌던 [표 4]가 여기서는 한 행(분야 하나)이 한 줄에 모였다. 한 칸 안의 줄바꿈은 `<br>`로 표시된다.

### 2-2. 변환 결과 정리

변환 결과에는 그림 자리 표시, `<br>` 태그, 특수 글머리표(`Ÿ`), 쪽마다 반복되는 머리말이 섞여 있다. 실제로 출력해 보고 규칙을 정한다(Day12 4-2와 같은 원칙).

In [11]:
# 변환결과 검토후 가비지 제거
def clean_md(text):
    text = re.sub(r"\*\*==> picture \[[^\]]+\] intentionally omitted <==\*\*", "", text)   # 그림 자리 표시
    text = text.replace("**----- Start of picture text -----**<br>", "").replace("**----- End of picture text -----**<br>", "")
    text = text.replace("<br>", " ")                                   # 표 칸 안의 줄바꿈
    text = text.replace("Ÿ", "")                                       # 특수 글머리표
    text = re.sub(r"토픽 분석을 통한 AI 주요 트렌드 및 2026 전망\s*\d*", "", text)   # 반복 머리말
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [12]:
before = md_pages[9]['text']
after = clean_md(before)
len(before), len(after)

(798, 746)

In [13]:
 # TODO: 표 칸 안의 <br>을 공백으로, 특수 글머리표 Ÿ를 빈 문자열로 바꾸세요
md_docs = [Document(page_content=clean_md(p["text"]),
                    metadata={"source": PDF.name, "page": p["metadata"]["page_number"]})
           for p in md_pages]

### 2-3. Markdown 저장소 만들기

Markdown은 표 한 덩어리가 길어서 조각을 조금 크게 자른다(800자, 겹침 120).

In [14]:
# 스플리터
md_chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120).split_documents(md_docs)

for i, c in enumerate(md_chunks, 1):
    c.metadata.update(chunk_id=i, doc_type="markdown")

# 스토어 생성
md_store = PGVector(embeddings=embeddings, collection_name="ai_trend_md",
                    connection=engine, use_jsonb=True, pre_delete_collection=True)
# 스토어 적재
md_store.add_documents(md_chunks)

['4e936e55-a99e-45d0-a041-010a297c514b',
 '518c1360-5e8a-43d6-92ad-137fb00cf144',
 'f99ec2dc-6cd1-4e02-9c25-c72af382afea',
 'ef085c06-0370-426f-bfda-f5882686f7ff',
 'be6e785b-7d47-416e-9ee2-be5984ff25e7',
 '55aa000b-266d-42b7-8bd4-bd217bf50673',
 '07410e06-002e-4406-8e29-4b93c4f70acb',
 '625413b1-0689-483c-9ea1-6827c9e11625',
 '8939e091-e050-4e12-b6c6-2fc2526ea5c1',
 '6177c435-2439-4001-af10-9a933b2227bc',
 '3255dac6-3da9-44a1-a539-3b31321653a7',
 'b67eb018-0aec-4a4c-8129-2d8b4e892d08',
 '69be9c50-ea60-488d-9cd4-1ccae00a7284',
 'b4d67eb2-007a-4ff3-9498-854e4cd68d59',
 'dcdb8caa-fc75-4862-81be-7ab48be2caca',
 'e2db0f5a-1318-48f6-8cee-844a50d133f8',
 '8df295c8-d950-46ea-bb77-44232be889fe',
 'd9642034-7eac-482c-bab3-3b7134e11053',
 '9a4fa7da-427b-43c7-80c1-06a7d8993a6c',
 '4f646e7b-249f-41d3-8ca4-49592395f509',
 'a054b5f7-0600-4dc2-b2a5-8523c8bc2f99',
 '13d1e701-f22f-447a-b94d-365971c8b542',
 '7750291f-890c-4f95-98cb-cd9bbc38b5a9',
 '8ef87a30-d443-4605-af18-878adf0a3020',
 'e99b3a9d-4ca0-

### 2-4. 같은 질문으로 비교 — 검색

In [15]:
def compare(query, k=3):
    print(f"[질문] {query}")
    for name, store in [("PDF", pdf_store), ("Markdown", md_store)]:
        top = store.similarity_search(query, k=k)
        pages = [d.metadata["page"] for d in top]
        print(f"  {name:<8} 쪽 {pages}")
    print()

for q in ["산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
          "AI 기술 분야 토픽모델링 결과에서 도출된 토픽들과 그 시사점은 무엇인가요?",
          "토픽모델링 전에 제거한 전처리 용어 유형과 예시는 무엇인가요?",
          "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"]:
    compare(q)

[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
  PDF      쪽 [10, 10, 9]
  Markdown 쪽 [10, 9, 4]

[질문] AI 기술 분야 토픽모델링 결과에서 도출된 토픽들과 그 시사점은 무엇인가요?
  PDF      쪽 [12, 14, 22]
  Markdown 쪽 [12, 16, 20]

[질문] 토픽모델링 전에 제거한 전처리 용어 유형과 예시는 무엇인가요?
  PDF      쪽 [20, 21, 11]
  Markdown 쪽 [11, 11, 21]

[질문] 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
  PDF      쪽 [18, 5, 19]
  Markdown 쪽 [5, 17, 18]



### 2-5. 같은 질문으로 비교 — 답변

In [16]:

q = "AI 기술 분야 토픽모델링 결과에서 도출된 토픽들과 그 시사점은 무엇인가요?"


### 관찰

| 확인 | PDF | Markdown |
| --- | --- | --- |
| [표 4]의 모양 | 칸과 칸 안의 글이 한 줄씩 흩어진다 | 한 행이 한 줄에 모인다 |
| [표 4] 질문의 검색 | 12·14·22쪽 (표가 있는 16쪽 없음) | 12·16·20쪽 (16쪽 포함) |
| [표 4] 질문의 답 | 핵심 이슈를 토픽으로 잘못 나눈다 | 토픽명·주요 키워드·핵심 이슈를 나눠 답한다 |
| 조각 수 | 더 많다 (500자) | 더 적다 (800자) |

> **주의:** Markdown 변환이 모든 문서에서 낫지는 않다. 표가 선 없이 그려진 문서나 스캔 PDF는 변환 결과부터 확인한다. 실습 4에서 두 저장소를 같은 평가 세트로 재 본다.

## 확인 문제

1. Naive RAG의 한계 세 가지(표현 편차, 표 깨짐, 무조건 k개)를 이 노트북의 출력으로 설명하면?
2. `similarity_search_with_score`의 점수는 높을수록 가까운가, 낮을수록 가까운가?
3. 보고서에 없는 질문에도 검색 결과가 3개 나오는 이유는?
4. 단순한 [표 2]는 PDF로도 답을 맞혔는데 [표 4]는 왜 못 맞혔나? Markdown에서는 무엇이 달라졌나?
5. Markdown 조각을 PDF 조각보다 크게 자른 이유는?